# Temă de Programare: Conductă de Imputare Pregătită pentru Producție

Bun venit la tema **Conducta de Imputare Pregatita pentru Productie**!

In productie, imputarea nu poate fi un pas executat o singura data, ci trebuie sa fie o componenta de prim rang a pipeline-ului de preprocesare. Asta inseamna:

- **Incapsulare**: logica de imputare este pusa intr-o clasa cu un API clar `fit/transform`, astfel incat sa se poata integra natural cu sklearn
- **Prevenirea scurgerii de informatie**: statisticile invatate in `fit()` sunt inghetate; `transform()` aplica doar valorile stocate
- **Robustete la cazuri-limita**: coloanele complet goale si coloanele constante trebuie detectate si tratate corect inainte de modelare
- **Auditabilitate**: fiecare decizie de imputare trebuie jurnalizata (numele coloanei, strategia, valoarea folosita, celulele imputate)
- **Serializare**: pipeline-ul antrenat poate fi salvat si reincarcat cu `joblib` pentru inferenta

**Ce vei face in aceasta tema:**

* Vei construi de la zero o clasa `ConfigurableImputer` cu `fit/transform/fit_transform`
* Vei scrie un detector de cazuri-limita pentru coloane patologice
* Vei implementa un `SklearnImputer` compatibil cu sklearn (mostenind `BaseEstimator`, `TransformerMixin`)
* Vei implementa un `ProductionImputer` care impune prevenirea stricta a scurgerii de informatie
* Vei construi un `ImputationLogger` care inregistreaza fiecare decizie de imputare
* Vei asambla un pipeline complet de preprocesare cu `ColumnTransformer` si `Pipeline`

Sa incepem!


---
<a name='submission'></a>

<h4 style="color:green; font-weight:bold;">SFATURI PENTRU EVALUAREA CU SUCCES A TEMEI TALE:</h4>

* Toate celulele sunt blocate cu excepția celor în care trebuie să trimiți soluțiile sau atunci când este menționat explicit că poți interacționa cu ele.

* În fiecare celulă de exercițiu, caută comentariile `### ÎNCEPUT DE COD AICI ###` și `### SFÂRȘIT DE COD AICI ###`. Acestea îți arată unde să scrii codul soluției. **Nu adăuga sau modifica niciun cod care se află în afara acestor comentarii**.

* Poți adăuga celule noi pentru a experimenta, dar acestea vor fi omise de evaluator, deci nu te baza pe celule create nou pentru a găzdui codul soluției — folosește locurile prevăzute în acest scop.

* **Evită utilizarea variabilelor globale** cu excepția cazului în care este absolut necesar. Evaluatorul testează codul tău într-un mediu izolat.

---

## Cuprins
- [Importuri](#imports)
- [1 - Clasa Configurable Imputer](#1---configurable-imputer-class)
    - **[Exercitiul 1 - `ConfigurableImputer`](#exercise-1---configurableimputer)**
- [2 - Gestionarea Cazurilor Limita](#2---handling-edge-cases)
    - **[Exercitiul 2 - `handle_edge_cases`](#exercise-2---handle_edge_cases)**
- [3 - Integrarea in Pipeline-ul Sklearn](#3---sklearn-pipeline-integration)
    - **[Exercitiul 3 - `SklearnImputer`](#exercise-3---sklearnimputer)**
- [4 - Tiparul Fit/Transform](#4---fittransform-pattern)
    - **[Exercitiul 4 - `ProductionImputer`](#exercise-4---productionimputer)**
- [5 - Jurnalizarea Imputarii](#5---imputation-logging)
    - **[Exercitiul 5 - `ImputationLogger`](#exercise-5---imputationlogger)**
- [6 - Pipeline Complet de Preprocesare](#6---complete-preprocessing-pipeline)
    - **[Exercitiul 6 - `create_preprocessing_pipeline`](#exercise-6---create_preprocessing_pipeline)**


<a name='imports'></a>
## Importuri

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer, make_column_selector
import joblib

In [ ]:
import helper_utils
import unittests

## Pregatirea Datelor

Folosim `helper_utils.generate_train_test_data()` pentru a produce un set de date cu tipuri mixte (numeric + categorial) si valori lipsa, reprezentativ pentru un scenariu real de ingestie.


In [ ]:
X_train, X_test, y_train, y_test = helper_utils.generate_train_test_data(random_state=42)

print(f"Training set:  {X_train.shape}")
print(f"Test set:      {X_test.shape}")
print(f"\nDtypes:\n{X_train.dtypes}")
print(f"\nMissing (train):\n{X_train.isnull().sum()}")

In [ ]:
helper_utils.visualize_missing_patterns(X_train)

<a name='1---configurable-imputer-class'></a>
## 1 - Clasa Imputer Configurabil

Incepem prin a construi un imputator **pure Python**, fara sa ne bazam pe sklearn. Acest lucru consolideaza mecanica tiparului fit/transform: `fit()` citeste datele de antrenare si stocheaza statisticile; `transform()` aplica valorile stocate pe orice set de date, fara sa mai atinga datele de antrenare.


<a name='exercise-1---configurableimputer'></a>
### **Exercitiul 1 - `ConfigurableImputer`**

**Sarcina ta:**

Implementeaza o clasa `ConfigurableImputer` cu urmatorul API:
- `__init__(strategy='mean', fill_value=None)` — stocheaza strategia
- `fit(X)` — calculeaza si stocheaza statisticile de imputare in `self.statistics_` (dict care mapeaza numele coloanei → valoarea de completare); returneaza `self`
- `transform(X)` — returneaza o copie a lui X cu valorile lipsa completate folosind `self.statistics_`; ridica `ValueError` daca `fit()` nu a fost apelata
- `fit_transform(X)` — wrapper convenabil: `self.fit(X).transform(X)`

Strategii suportate: `'mean'`, `'median'`, `'mode'`, `'constant'`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
def fit(self, X):
    df = pd.DataFrame(X)
    self.statistics_ = {}
    for col in df.columns:
        if self.strategy == 'mean':
            self.statistics_[col] = df[col].mean()
        elif self.strategy == 'median':
            self.statistics_[col] = df[col].median()
        elif self.strategy == 'mode':
            self.statistics_[col] = df[col].mode()[0]
        else:  # constant
            self.statistics_[col] = self.fill_value
    return self

def transform(self, X):
    if not hasattr(self, 'statistics_'):
        raise ValueError('Imputer has not been fitted yet. Call fit() first.')
    result = pd.DataFrame(X).copy()
    for col, value in self.statistics_.items():
        if col in result.columns:
            result[col] = result[col].fillna(value)
    return result
```

</details>


In [ ]:
# GRADED CLASS: ConfigurableImputer
class ConfigurableImputer:
    """
    A configurable imputer that supports mean, median, mode, and constant strategies.

    After calling fit(X), self.statistics_ is a dict mapping column name to fill value.
    """

    ### ÎNCEPUT DE COD AICI ###

    def __init__(self, strategy='mean', fill_value=None):
        self.strategy   = None  # store strategy
        self.fill_value = None  # store fill_value

    def fit(self, X):
        """
        Compute and store imputation statistics from X.

        Args:
            X (pd.DataFrame or array): Training data.
        Returns:
            self
        """
        df = pd.DataFrame(X)
        self.statistics_ = {}
        for col in df.columns:
            pass  # fill in statistics_[col] based on self.strategy
        return self

    def transform(self, X):
        """
        Apply stored statistics to fill missing values.

        Args:
            X (pd.DataFrame or array): Data to transform.
        Returns:
            pd.DataFrame: Imputed copy of X.
        Raises:
            ValueError: if fit() has not been called.
        """
        if not hasattr(self, 'statistics_'):
            raise ValueError('Imputer has not been fitted yet. Call fit() first.')
        result = pd.DataFrame(X).copy()
        # apply statistics_
        return result

    def fit_transform(self, X):
        """
        Fit on X and immediately transform it.
        """
        return self.fit(X).transform(X)

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
numeric_train = X_train.select_dtypes(include='number')

ci = ConfigurableImputer(strategy='mean')
ci.fit(numeric_train)
print("Learned statistics (mean):")
for col, val in ci.statistics_.items():
    print(f"  {col}: {val:.4f}" if isinstance(val, float) else f"  {col}: {val}")

imputed = ci.transform(numeric_train)
print(f"\nMissing after transform: {imputed.isnull().sum().sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_1(ConfigurableImputer)

<a name='2---handling-edge-cases'></a>
## 2 - Gestionarea Cazurilor Limita

Datele din productie contin frecvent coloane patologice: coloane complet goale (imputatorul ar completa cu 0 sau NaN — lipsit de sens) si coloane constante (varianta zero — imputarea este valida, dar coloana nu contine informatie). Detectarea si marcarea acestor coloane inainte de modelare previn erori silentioase.


<a name='exercise-2---handle_edge_cases'></a>
### **Exercitiul 2 - `handle_edge_cases`**

**Sarcina ta:**

Implementeaza `handle_edge_cases(col)` astfel incat sa inspecteze un `pd.Series` si sa returneze un dictionar de diagnostic.

**Cerinte:**
- Returneaza un `dict` cu exact aceste chei:
  - `'is_valid'` — `True` daca coloana poate fi imputata fara probleme
  - `'issue_type'` — `None` daca este valida; `'all_missing'` daca toate valorile sunt NaN; `'constant'` daca toate valorile non-NaN sunt identice

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
if col.isnull().all():
    return {'is_valid': False, 'issue_type': 'all_missing'}
if col.dropna().nunique() == 1:
    return {'is_valid': False, 'issue_type': 'constant'}
return {'is_valid': True, 'issue_type': None}
```

</details>


In [ ]:
# GRADED FUNCTION: handle_edge_cases
def handle_edge_cases(col):
    """
    Detect pathological column patterns that need special treatment.

    Args:
        col (pd.Series): A single column of a DataFrame.

    Returns:
        dict: {'is_valid': bool, 'issue_type': None | 'all_missing' | 'constant'}
    """
    ### ÎNCEPUT DE COD AICI ###

    # Check for all-missing column
    # Check for constant column (all non-null values are the same)
    # Otherwise return valid

    return {'is_valid': None, 'issue_type': None}

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
all_missing_col  = pd.Series([np.nan, np.nan, np.nan])
constant_col     = pd.Series([3.0, 3.0, np.nan, 3.0])
normal_col       = pd.Series([1.0, 2.0, np.nan, 4.0])

print(handle_edge_cases(all_missing_col))   # {'is_valid': False, 'issue_type': 'all_missing'}
print(handle_edge_cases(constant_col))      # {'is_valid': False, 'issue_type': 'constant'}
print(handle_edge_cases(normal_col))        # {'is_valid': True,  'issue_type': None}

In [ ]:
# Testează-ți codul!
unittests.exercise_2(handle_edge_cases)

<a name='3---sklearn-pipeline-integration'></a>
## 3 - Integrarea in Pipeline-ul Sklearn

Pentru a se integra in `Pipeline` si `ColumnTransformer` din sklearn, un transformator trebuie sa mosteneasca `BaseEstimator` si `TransformerMixin`. Mostenirea `TransformerMixin` ofera gratuit `fit_transform()`, iar `BaseEstimator` ofera `get_params()` / `set_params()`, facand posibila folosirea grid search-ului.


<a name='exercise-3---sklearnimputer'></a>
### **Exercitiul 3 - `SklearnImputer`**

**Sarcina ta:**

Implementeaza `SklearnImputer(strategy='mean')` astfel incat sa mosteneasca atat `BaseEstimator`, cat si `TransformerMixin`.

**Cerinte:**
- `__init__(self, strategy='mean')` — stocheaza `strategy` in `self.strategy`
- `fit(self, X, y=None)` — calculeaza valorile de completare pe coloane, le stocheaza in `self.fill_values_` (dict) si returneaza `self`
- `transform(self, X)` — aplica `self.fill_values_` si returneaza un `pd.DataFrame`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

**Cheie:** Cand mostenesti `BaseEstimator`, **nu** apela `super().__init__()` cu argumente. Stocheaza toti parametrii ca atribute simple cu exact acelasi nume ca in `__init__` (conventie folosita de `get_params()` din sklearn).

```python
class SklearnImputer(BaseEstimator, TransformerMixin):
    def __init__(self, strategy='mean'):
        self.strategy = strategy  # MUST match __init__ param name!

    def fit(self, X, y=None):
        df = pd.DataFrame(X)
        self.fill_values_ = {}
        for col in df.columns:
            if self.strategy == 'mean':   self.fill_values_[col] = df[col].mean()
            elif self.strategy == 'median': self.fill_values_[col] = df[col].median()
            else: self.fill_values_[col] = df[col].mode()[0]
        return self

    def transform(self, X):
        df = pd.DataFrame(X).copy()
        for col, val in self.fill_values_.items():
            if col in df.columns:
                df[col] = df[col].fillna(val)
        return df
```

</details>


In [ ]:
# GRADED CLASS: SklearnImputer
class SklearnImputer(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible imputer. Inheriting TransformerMixin gives fit_transform() for free.
    Inheriting BaseEstimator gives get_params() / set_params() for grid search.
    """

    ### ÎNCEPUT DE COD AICI ###

    def __init__(self, strategy='mean'):
        self.strategy = None  # store strategy

    def fit(self, X, y=None):
        """
        Compute and store fill values per column.

        Returns:
            self
        """
        df = pd.DataFrame(X)
        self.fill_values_ = {}
        for col in df.columns:
            pass  # compute fill value for each column
        return self

    def transform(self, X):
        """
        Fill missing values using stored fill_values_.

        Returns:
            pd.DataFrame: Imputed copy of X.
        """
        df = pd.DataFrame(X).copy()
        # apply fill_values_
        return df

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
si = SklearnImputer(strategy='median')
X_num_train = X_train.select_dtypes(include='number')
result = si.fit_transform(X_num_train)  # fit_transform is free from TransformerMixin
print(f"Missing after transform: {result.isnull().sum().sum()}")
print(f"get_params():            {si.get_params()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_3(SklearnImputer)

<a name='4---fittransform-pattern'></a>
## 4 - Tiparul Fit/Transform

`ProductionImputer` impune **prevenirea stricta a scurgerii de informatie** prin ridicarea unui `RuntimeError` daca `transform()` este apelata inainte de `fit()`. De asemenea, urmareste daca imputatorul a fost antrenat, astfel incat dublul fit accidental (care ar suprascrie in tacere statisticile invatate) sa poata fi detectat.


<a name='exercise-4---productionimputer'></a>
### **Exercitiul 4 - `ProductionImputer`**

**Sarcina ta:**

Implementeaza `ProductionImputer(strategy='mean')` astfel incat:
- Sa urmareasca daca `fit()` a fost apelata prin `self.is_fitted_`
- Sa ridice `RuntimeError('Imputer not fitted. Call fit() before transform.')` daca `transform()` este apelata inainte de `fit()`
- `fit()` sa invete statisticile si sa seteze `self.is_fitted_ = True`; sa returneze `self`
- `transform()` sa aplice statisticile stocate; sa returneze un `pd.DataFrame`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
def __init__(self, strategy='mean'):
    self.strategy   = strategy
    self.is_fitted_ = False

def fit(self, X):
    df = pd.DataFrame(X)
    self.statistics_ = {}
    for col in df.columns:
        self.statistics_[col] = df[col].mean()  # simplified
    self.is_fitted_ = True
    return self

def transform(self, X):
    if not self.is_fitted_:
        raise RuntimeError('Imputer not fitted. Call fit() before transform.')
    ...
```

</details>


In [ ]:
# GRADED CLASS: ProductionImputer
class ProductionImputer:
    """
    An imputer that strictly enforces the fit-before-transform pattern.
    Raises RuntimeError if transform() is called before fit().
    """

    ### ÎNCEPUT DE COD AICI ###

    def __init__(self, strategy='mean'):
        self.strategy   = None  # store strategy
        self.is_fitted_ = None  # set to False initially

    def fit(self, X):
        """
        Learn statistics from X. Sets self.is_fitted_ = True.
        Returns:
            self
        """
        df = pd.DataFrame(X)
        self.statistics_ = {}
        for col in df.columns:
            pass  # compute fill value based on self.strategy
        None  # set is_fitted_ = True
        return self

    def transform(self, X):
        """
        Fill missing values. Raises RuntimeError if not fitted.
        Returns:
            pd.DataFrame: Imputed copy of X.
        """
        if not self.is_fitted_:
            raise RuntimeError('Imputer not fitted. Call fit() before transform.')
        df = pd.DataFrame(X).copy()
        # apply statistics_
        return df

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
pi = ProductionImputer(strategy='mean')
print(f"is_fitted_ before fit: {pi.is_fitted_}")

# Should raise RuntimeError
try:
    pi.transform(X_num_train)
    print("ERROR: Should have raised RuntimeError!")
except RuntimeError as e:
    print(f"Correctly raised RuntimeError: {e}")

pi.fit(X_num_train)
print(f"is_fitted_ after fit:  {pi.is_fitted_}")
out = pi.transform(X_num_train)
print(f"Missing after transform: {out.isnull().sum().sum()}")

In [ ]:
# Testează-ți codul!
unittests.exercise_4(ProductionImputer)

<a name='5---imputation-logging'></a>
## 5 - Jurnalizarea Imputarii

In productie, **orice transformare de date trebuie sa poata fi auditata**. `ImputationLogger` inregistreaza o intrare pentru fiecare coloana imputata, astfel incat data scientistii sa poata inspecta ulterior ce valori au fost inlocuite, cate celule au fost afectate si daca strategia de imputare pare rezonabila pentru distributia datelor.


<a name='exercise-5---imputationlogger'></a>
### **Exercitiul 5 - `ImputationLogger`**

**Sarcina ta:**

Implementeaza o clasa `ImputationLogger` care **atat imputeaza valorile lipsa, cat si jurnalizeaza fiecare imputare individuala**:
- `__init__(strategy='mean')` — stocheaza strategia; initializeaza `self.imputation_log_` ca `pd.DataFrame` gol
- `fit(X)` — invata statisticile de completare pe coloane in `self.statistics_` (dict ce mapeaza numele coloanei → valoarea folosita la completare); returneaza `self`
- `transform(X)` — completeaza valorile lipsa **si** jurnalizeaza fiecare celula imputata; stocheaza jurnalul complet ca `pd.DataFrame` in `self.imputation_log_`; returneaza `pd.DataFrame`-ul imputat
- `fit_transform(X)` — apeleaza `self.fit(X).transform(X)` si returneaza rezultatul

Fiecare rand din `self.imputation_log_` trebuie sa contina exact aceste chei:
- `'row'` — indexul randului in care a fost imputata celula
- `'column'` — numele coloanei
- `'fill_value'` — valoarea substituita
- `'strategy'` — strategia de imputare folosita

Strategii suportate: `'mean'`, `'median'`, `'mode'`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
def transform(self, X):
    df = pd.DataFrame(X).copy()
    log_entries = []
    for col in df.columns:
        fill_val = self.statistics_[col]
        missing_rows = df.index[df[col].isnull()].tolist()
        for row_idx in missing_rows:
            log_entries.append({'row': row_idx, 'column': col,
                                'fill_value': fill_val, 'strategy': self.strategy})
        df[col] = df[col].fillna(fill_val)
    self.imputation_log_ = pd.DataFrame(log_entries)
    return df
```

</details>


In [ ]:
# GRADED CLASS: ImputationLogger

class ImputationLogger:
    """
    Un imputer care jurnalizează fiecare valoare lipsă pe care o completează.

    După fit_transform(X), self.imputation_log_ este un pd.DataFrame cu
    un rând pe celulă imputată: {'row', 'column', 'fill_value', 'strategy'}.
    """

    ### ÎNCEPUT DE COD AICI ###

    def __init__(self, strategy='mean'):
        self.strategy = strategy
        self.imputation_log_ = None  # will become a pd.DataFrame after transform

    def fit(self, X):
        # Învață statisticile de completare pe coloană și stochează în self.statistics_
        pass

    def transform(self, X):
        # Imputează valorile lipsă și jurnalizează fiecare celulă imputată individual
        # Stochează jurnalul ca pd.DataFrame în self.imputation_log_
        pass

    def fit_transform(self, X):
        pass  # apelează fit apoi transform

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
X_num_with_missing = X_train.select_dtypes(include='number').copy()
logger = ImputationLogger(strategy='mean')
X_imputed = logger.fit_transform(X_num_with_missing)

print(f"Imputed shape:    {X_imputed.shape}")
print(f"Missing after:    {X_imputed.isnull().sum().sum()}")
print(f"\nImputation log ({len(logger.imputation_log_)} entries):")
print(logger.imputation_log_.head(10))

In [ ]:
# Testează-ți codul!
unittests.exercise_5(ImputationLogger)

<a name='6---complete-preprocessing-pipeline'></a>
## 6 - Pipeline Complet de Preprocesare

Un pipeline de productie leaga imputarea, scalarea si encodarea intr-un singur `Pipeline` sklearn care poate fi antrenat o singura data si serializat. Coloanele numerice si categoriale sunt tratate separat prin `ColumnTransformer`.


<a name='exercise-6---create_preprocessing_pipeline'></a>
### **Exercitiul 6 - `create_preprocessing_pipeline`**

**Sarcina ta:**

Implementeaza `create_preprocessing_pipeline(numeric_cols=None, cat_cols=None)` astfel incat sa returneze un `Pipeline` sklearn cu **cel putin 2 pasi**:
1. Pasul `'preprocessor'` — un `ColumnTransformer` care aplica:
   - Ramura numerica: `SimpleImputer(strategy='mean')` → `StandardScaler()`
   - Ramura categoriala: `SimpleImputer(strategy='most_frequent')` → `OneHotEncoder(handle_unknown='ignore')`
2. Pasul `'passthrough'` — un pas identitate `FunctionTransformer()`

**Cerinte:**
- Cand `numeric_cols=None`, detecteaza automat coloanele numerice folosind `make_column_selector(dtype_include=np.number)`
- Cand `cat_cols=None`, detecteaza automat coloanele categoriale folosind `make_column_selector(dtype_include=object)`
- Functia trebuie sa functioneze cand este apelata **fara argumente**: `create_preprocessing_pipeline()`

<details>
  <summary><b><font color="green">Indicatii suplimentare de cod (click pentru extindere)</font></b></summary>

```python
if numeric_cols is None:
    numeric_cols = make_column_selector(dtype_include=np.number)
if cat_cols is None:
    cat_cols = make_column_selector(dtype_include=object)

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='mean')),
                      ('scaler',  StandardScaler())]), numeric_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                      ('encoder', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('passthrough',  FunctionTransformer())
])
return pipeline
```

</details>


In [ ]:
# GRADED FUNCTION: create_preprocessing_pipeline
def create_preprocessing_pipeline(numeric_cols=None, cat_cols=None):
    """
    Construiește un pipeline complet de preprocesare pentru date de tip mixt.

    Numeric:     SimpleImputer(mean)           → StandardScaler
    Categorial:  SimpleImputer(most_frequent)  → OneHotEncoder

    Args:
        numeric_cols (list | None): Numele coloanelor numerice.
                                    Detectate automat când None.
        cat_cols     (list | None): Numele coloanelor categoriale.
                                    Detectate automat când None.

    Returns:
        sklearn.pipeline.Pipeline: Un pipeline cu 2 pași (preprocessor + passthrough).
    """
    ### ÎNCEPUT DE COD AICI ###

    # Detectează automat tipurile de coloane când nu sunt furnizate
    # Construiește transformatorul numeric: imputare → scalare
    # Construiește transformatorul categorial: imputare → encodare
    # Combină cu ColumnTransformer
    # Returnează un Pipeline cu 2 pași (preprocessor + FunctionTransformer passthrough)

    return None

    ### SFÂRȘIT DE COD AICI ###

In [ ]:
# Apelează fără argumente — tipurile de coloane sunt detectate automat din datele de antrenament
pipeline = create_preprocessing_pipeline()
pipeline.fit(X_train)

X_tr_processed = pipeline.transform(X_train)
X_te_processed = pipeline.transform(X_test)
print(f"Processed training shape: {X_tr_processed.shape}")
print(f"Pipeline steps:           {pipeline.steps}")

In [ ]:
# Testează-ți codul!
unittests.exercise_6(create_preprocessing_pipeline)

## Serializare cu Joblib

Dupa ce pipeline-ul a fost antrenat, salveaza-l cu `joblib.dump()` pentru a persista statisticile invatate. Incarca-l cu `joblib.load()` pentru a reproduce transformari identice la momentul servirii modelului.


In [ ]:
# Save and reload the fitted pipeline
joblib.dump(pipeline, '/tmp/imputation_pipeline.joblib')
loaded_pipeline = joblib.load('/tmp/imputation_pipeline.joblib')

X_te_loaded = loaded_pipeline.transform(X_test)
print(f"Loaded pipeline output shape: {X_te_loaded.shape}")
print("Outputs match:", np.allclose(X_tr_processed, pipeline.transform(X_train)))